# Billing, cash application, AR and GL

Inspect document-level billing, invoice allocations, one AR balance per invoice and cutoff, and GL posting lines. The cutoff comes from the Silver manifest. Reported-only AR remains distinguishable from calculated balances.

Run from the `gold/` directory after setting `silver_manifest` in `gold_settings.json`.

In [1]:
from pathlib import Path

from IPython.display import display
import pandas as pd

from gold_pipeline import build, publish

SETTINGS_PATH = Path("gold_settings.json")
gold = build(SETTINGS_PATH, stage="finance")
print("Silver release", gold.manifest["model_run_id"])
print("Cutoff", gold.settings["cutoff"])
display(pd.DataFrame([{"table": name, "rows": len(rows)} for name, rows in gold.tables.items()]))

Silver release 409a707f49d90b112334f8bb4bfd27423a7f3f48f7039eecdff871dddee256e5
Cutoff 2026-12-16


,table,rows
0,dim_legal_entity,2
1,dim_customer,5
2,dim_product,5
3,dim_date,13
4,dim_currency,1
5,fact_sales_order_booking,8
6,dim_warehouse,3
7,fact_shipment_line,5
8,dim_document_type,2
9,fact_billing_document,8


In [2]:
for name in ("fact_billing_document", "fact_payment_allocation", "fact_ar_snapshot", "fact_gl_posting"):
    print(name)
    display(pd.DataFrame(gold.records(name)).head(20))

fact_billing_document


,id,legal_entity_id,document_id,document_reference,original_invoice_id,sales_order_id,document_type,posting_date,due_date,currency,signed_billed_amount,document_count,document_date_key,invoice_due_date_key,customer_key,legal_entity_key,document_type_key,currency_key
0,"[""parent"",""ACC-SINV-2026-00001""]",parent,ACC-SINV-2026-00001,P-I1,None,"[""parent"",""SAL-ORD-2026-00001""]",invoice,2026-11-03,2026-12-03,INR,60000.000000000000,1,date:2026-11-03,date:2026-12-03,customer:G-CUST-001,legal_entity:parent,document_type:invoice,currency:INR
1,"[""parent"",""ACC-SINV-2026-00002""]",parent,ACC-SINV-2026-00002,P-I2,None,"[""parent"",""SAL-ORD-2026-00001""]",invoice,2026-11-05,2026-12-05,INR,40000.000000000000,1,date:2026-11-05,date:2026-12-05,customer:G-CUST-001,legal_entity:parent,document_type:invoice,currency:INR
2,"[""parent"",""ACC-SINV-2026-00003""]",parent,ACC-SINV-2026-00003,P-CR1,"[""parent"",""ACC-SINV-2026-00002""]",None,credit_note,2026-11-15,2026-11-15,INR,-5000.000000000000,1,date:2026-11-15,date:2026-11-15,customer:G-CUST-001,legal_entity:parent,document_type:credit_note,currency:INR
3,"[""parent"",""ACC-SINV-2026-00004""]",parent,ACC-SINV-2026-00004,P-BRONZE-R2-INVOICE,None,"[""parent"",""SAL-ORD-2026-00006""]",invoice,2026-11-15,2026-11-15,INR,20000.000000000000,1,date:2026-11-15,date:2026-11-15,customer:G-CUST-001,legal_entity:parent,document_type:invoice,currency:INR
4,"[""parent"",""ACC-SINV-2026-00005""]",parent,ACC-SINV-2026-00005,P-BRONZE-R2-CREDIT,"[""parent"",""ACC-SINV-2026-00004""]",None,credit_note,2026-11-15,2026-11-15,INR,-10000.000000000000,1,date:2026-11-15,date:2026-11-15,customer:G-CUST-001,legal_entity:parent,document_type:credit_note,currency:INR
5,"[""dealer"",""D-BRONZE-R2-CREDIT""]",dealer,D-BRONZE-R2-CREDIT,D-BRONZE-R2-CREDIT,None,"[""dealer"",""D-BRONZE-R2-ORDER""]",credit_note,2026-11-15,2026-12-15,INR,-10000.000000000000,1,date:2026-11-15,date:2026-12-15,customer:G-CUST-001,legal_entity:dealer,document_type:credit_note,currency:INR
6,"[""dealer"",""D-BRONZE-R2-INVOICE""]",dealer,D-BRONZE-R2-INVOICE,D-BRONZE-R2-INVOICE,None,"[""dealer"",""D-BRONZE-R2-ORDER""]",invoice,2026-11-15,2026-12-15,INR,20000.000000000000,1,date:2026-11-15,date:2026-12-15,customer:G-CUST-001,legal_entity:dealer,document_type:invoice,currency:INR
7,"[""dealer"",""D-I1""]",dealer,D-I1,D-I1,None,"[""dealer"",""D-S1""]",invoice,2026-11-05,2026-12-05,INR,40000.000000000000,1,date:2026-11-05,date:2026-12-05,customer:G-CUST-001,legal_entity:dealer,document_type:invoice,currency:INR


fact_payment_allocation


,id,payment_id,invoice_id,allocation_id,legal_entity_id,payment_date,application_date,invoice_due_date,allocated_cash_amount,allocation_count,currency,customer_key,payment_date_key,application_date_key,invoice_due_date_key,legal_entity_key,currency_key
0,"[""parent"",""ACC-PAY-2026-00001"",""vr1ekg0uio""]","[""parent"",""ACC-PAY-2026-00001""]","[""parent"",""ACC-SINV-2026-00001""]","[""parent"",""ACC-PAY-2026-00001"",""vr1ekg0uio""]",parent,2026-11-10,None,2026-12-03,25000.000000000000,1,INR,customer:G-CUST-001,date:2026-11-10,date:__unknown__,date:2026-12-03,legal_entity:parent,currency:INR
1,"[""dealer"",""D-A1""]","[""dealer"",""D-R1""]","[""dealer"",""D-I1""]","[""dealer"",""D-A1""]",dealer,2026-11-12,None,2026-12-05,10000.000000000000,1,INR,customer:G-CUST-001,date:2026-11-12,date:__unknown__,date:2026-12-05,legal_entity:dealer,currency:INR


fact_ar_snapshot


,id,invoice_id,legal_entity_id,as_of_date,invoice_date,due_date,customer_key,currency,open_amount,days_past_due,aging_bucket,balance_basis,coverage_status,as_of_date_key,invoice_date_key,invoice_due_date_key,legal_entity_key,currency_key,aging_bucket_key
0,"[""[\""dealer\"",\""D-BRONZE-R2-INVOICE\""]"",""2026-...","[""dealer"",""D-BRONZE-R2-INVOICE""]",dealer,2026-12-16,2026-11-15,2026-12-15,customer:G-CUST-001,INR,10000.000000000000,1,1–30,reported,reported_only,date:2026-12-16,date:2026-11-15,date:2026-12-15,legal_entity:dealer,currency:INR,aging_bucket:1–30
1,"[""[\""dealer\"",\""D-I1\""]"",""2026-12-16"",""INR""]","[""dealer"",""D-I1""]",dealer,2026-12-16,2026-11-05,2026-12-05,customer:G-CUST-001,INR,30000.000000000000,11,1–30,calculated,calculated_only,date:2026-12-16,date:2026-11-05,date:2026-12-05,legal_entity:dealer,currency:INR,aging_bucket:1–30
2,"[""[\""parent\"",\""ACC-SINV-2026-00001\""]"",""2026-...","[""parent"",""ACC-SINV-2026-00001""]",parent,2026-12-16,2026-11-03,2026-12-03,customer:G-CUST-001,INR,35000.000000000000,13,1–30,calculated,calculated_only,date:2026-12-16,date:2026-11-03,date:2026-12-03,legal_entity:parent,currency:INR,aging_bucket:1–30
3,"[""[\""parent\"",\""ACC-SINV-2026-00002\""]"",""2026-...","[""parent"",""ACC-SINV-2026-00002""]",parent,2026-12-16,2026-11-05,2026-12-05,customer:G-CUST-001,INR,35000.000000000000,11,1–30,calculated,calculated_only,date:2026-12-16,date:2026-11-05,date:2026-12-05,legal_entity:parent,currency:INR,aging_bucket:1–30
4,"[""[\""parent\"",\""ACC-SINV-2026-00004\""]"",""2026-...","[""parent"",""ACC-SINV-2026-00004""]",parent,2026-12-16,2026-11-15,2026-11-15,customer:G-CUST-001,INR,10000.000000000000,31,31–60,calculated,calculated_only,date:2026-12-16,date:2026-11-15,date:2026-11-15,legal_entity:parent,currency:INR,aging_bucket:31–60


fact_gl_posting


,id,legal_entity_id,posting_date,currency,gl_entry_id,voucher_id,voucher_type,billing_document_id,account_source_id,reporting_category,debit_amount,credit_amount,gl_line_count,posting_date_key,legal_entity_key,gl_account_key,account_category_key,currency_key
0,"[""parent"",""09029f86af""]",parent,2026-11-15,INR,09029f86af,MAT-STE-2026-00003,Stock Entry,None,"[""parent"",""Stock In Hand - OPD""]",INVENTORY,12000.000000000000,0E-12,1,date:2026-11-15,legal_entity:parent,"gl_account:[""parent"",""Stock In Hand - OPD""]",reporting_account_category:INVENTORY,currency:INR
1,"[""parent"",""3b7be7fe71""]",parent,2026-11-15,INR,3b7be7fe71,MAT-DN-2026-00003,Delivery Note,None,"[""parent"",""Stock In Hand - OPD""]",INVENTORY,0E-12,12000.000000000000,1,date:2026-11-15,legal_entity:parent,"gl_account:[""parent"",""Stock In Hand - OPD""]",reporting_account_category:INVENTORY,currency:INR
2,"[""parent"",""6eb02aaaa3""]",parent,2026-11-15,INR,6eb02aaaa3,ACC-SINV-2026-00004,Sales Invoice,"[""parent"",""ACC-SINV-2026-00004""]","[""parent"",""Sales - OPD""]",SALES_REVENUE,0E-12,20000.000000000000,1,date:2026-11-15,legal_entity:parent,"gl_account:[""parent"",""Sales - OPD""]",reporting_account_category:SALES_REVENUE,currency:INR
3,"[""parent"",""7acfb1ad2a""]",parent,2026-11-15,INR,7acfb1ad2a,MAT-DN-2026-00003,Delivery Note,None,"[""parent"",""Cost of Goods Sold - OPD""]",COGS,12000.000000000000,0E-12,1,date:2026-11-15,legal_entity:parent,"gl_account:[""parent"",""Cost of Goods Sold - OPD""]",reporting_account_category:COGS,currency:INR
4,"[""parent"",""93614da8ce""]",parent,2026-11-15,INR,93614da8ce,ACC-SINV-2026-00005,Sales Invoice,"[""parent"",""ACC-SINV-2026-00005""]","[""parent"",""Sales - OPD""]",SALES_REVENUE,10000.000000000000,0E-12,1,date:2026-11-15,legal_entity:parent,"gl_account:[""parent"",""Sales - OPD""]",reporting_account_category:SALES_REVENUE,currency:INR
5,"[""parent"",""981f76fdd4""]",parent,2026-11-15,INR,981f76fdd4,ACC-SINV-2026-00004,Sales Invoice,"[""parent"",""ACC-SINV-2026-00004""]","[""parent"",""Debtors - OPD""]",TRADE_AR,20000.000000000000,0E-12,1,date:2026-11-15,legal_entity:parent,"gl_account:[""parent"",""Debtors - OPD""]",reporting_account_category:TRADE_AR,currency:INR
6,"[""parent"",""ACC-GLE-2026-00001""]",parent,2026-11-01,INR,ACC-GLE-2026-00001,MAT-STE-2026-00001,Stock Entry,None,"[""parent"",""Stock In Hand - OPD""]",INVENTORY,36000.000000000000,0E-12,1,date:2026-11-01,legal_entity:parent,"gl_account:[""parent"",""Stock In Hand - OPD""]",reporting_account_category:INVENTORY,currency:INR
7,"[""parent"",""ACC-GLE-2026-00002""]",parent,2026-11-01,INR,ACC-GLE-2026-00002,MAT-STE-2026-00001,Stock Entry,None,"[""parent"",""Cost of Goods Sold - OPD""]",COGS,0E-12,36000.000000000000,1,date:2026-11-01,legal_entity:parent,"gl_account:[""parent"",""Cost of Goods Sold - OPD""]",reporting_account_category:COGS,currency:INR
8,"[""parent"",""ACC-GLE-2026-00003""]",parent,2026-11-01,INR,ACC-GLE-2026-00003,MAT-STE-2026-00002,Stock Entry,None,"[""parent"",""Stock In Hand - OPD""]",INVENTORY,24000.000000000000,0E-12,1,date:2026-11-01,legal_entity:parent,"gl_account:[""parent"",""Stock In Hand - OPD""]",reporting_account_category:INVENTORY,currency:INR
9,"[""parent"",""ACC-GLE-2026-00004""]",parent,2026-11-01,INR,ACC-GLE-2026-00004,MAT-STE-2026-00002,Stock Entry,None,"[""parent"",""Cost of Goods Sold - OPD""]",COGS,0E-12,24000.000000000000,1,date:2026-11-01,legal_entity:parent,"gl_account:[""parent"",""Cost of Goods Sold - OPD""]",reporting_account_category:COGS,currency:INR


In [3]:
display(pd.DataFrame(gold.checks))
display(pd.DataFrame(gold.exceptions))

,rule,expected,actual,passed
0,bookings_preserve_eligible_line_count,8,8,True
1,bookings_preserve_eligible_line_amount,220000.000000000000,220000.000000000000,True
2,baseline_booking_rows,2,2,True
3,baseline_distinct_orders,2,2,True
4,baseline_booking_inr,140000,140000.000000000000,True
5,baseline_shared_customer_product,1,1,True
6,baseline_pumps_each,11,11.000000000000,True


,rule,source_id
0,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00002"",""vqk0j2tpqq""]"
1,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00003"",""vqltad6jqq""]"
2,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00004"",""vqlfquq375""]"
3,booking_not_eligible,"[""parent"",""SAL-ORD-2026-00005"",""vqm83vc86c""]"
